# Getting started with MCP using the FastMCP package 

A notebook to test out the FastMCP python package. 

We'll set up functions and we will set up the MCP connections.

Then we will use Ollama to run models locally. 
In a previous notebook you have worked with Ollama, if not see there for more info.

## 0. Installs and imports


In [ ]:
import platform
platform.python_version()

In [ ]:
!pip install fastmcp

## 1. Use a single call to get the weather in Amsterdam

In [ ]:
import requests
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Weather Service")

def get_coordinates(city: str) -> tuple[float, float]:
    """Convert a city name to lat/lon using Open-Meteo's geocoding API."""
    url = "https://geocoding-api.open-meteo.com/v1/search"
    response = requests.get(url, params={"name": city, "count": 1})
    data = response.json()
    if not data.get("results"):
        raise ValueError(f"City '{city}' not found")
    result = data["results"][0]
    return result["latitude"], result["longitude"]

@mcp.tool()
def get_forecast(city: str, days: int = 3) -> str:
    """Get a weather forecast for a city. Days can be 1–7."""
    lat, lon = get_coordinates(city)
    
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": ["temperature_2m_max", "temperature_2m_min", "precipitation_sum", "weathercode"],
        "forecast_days": min(days, 7),
        "timezone": "auto"
    }
    response = requests.get(url, params=params)
    data = response.json()["daily"]

    lines = [f"Forecast for {city}:\n"]
    for i in range(len(data["time"])):
        lines.append(
            f"{data['time'][i]}  "
            f"↑{data['temperature_2m_max'][i]}°C  "
            f"↓{data['temperature_2m_min'][i]}°C  "
            f"Rain: {data['precipitation_sum'][i]}mm"
        )
    return "\n".join(lines)

# Try it
print(get_forecast("Amsterdam", days=3))

### Reflection question

Was MCP here the right choice? Or could you have used REST as well??

## 2. Plan a trip to Paris with MCP

In [ ]:
import requests
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Weekend Trip Planner")

# --- helper (reused from before) ---
def get_coordinates(city: str) -> tuple[float, float]:
    url = "https://geocoding-api.open-meteo.com/v1/search"
    r = requests.get(url, params={"name": city, "count": 1})
    result = r.json()["results"][0]
    return result["latitude"], result["longitude"]


# --- Tool 1: Weekend weather ---
@mcp.tool()
def get_weekend_rain(city: str) -> str:
    """Check whether it will rain this weekend (Saturday + Sunday) in a city."""
    lat, lon = get_coordinates(city)

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": ["precipitation_sum", "weathercode"],
        "forecast_days": 7,
        "timezone": "auto"
    }
    data = requests.get(url, params=params).json()["daily"]

    # Find Saturday and Sunday
    from datetime import date, timedelta
    today = date.today()
    days_until_saturday = (5 - today.weekday()) % 7
    saturday = today + timedelta(days=days_until_saturday)
    sunday = saturday + timedelta(days=1)
    weekend_dates = {str(saturday), str(sunday)}

    results = []
    for i, day in enumerate(data["time"]):
        if day in weekend_dates:
            rain = data["precipitation_sum"][i]
            results.append(f"{day}: {rain}mm of rain")

    if not results:
        return "Could not find weekend forecast."
    
    total_rain = sum(
        data["precipitation_sum"][i]
        for i, day in enumerate(data["time"])
        if day in weekend_dates
    )
    return "\n".join(results)


# --- Tool 2: Train ticket price ---
@mcp.tool()
def get_train_price(origin: str, destination: str) -> str:
    """Get the cheapest train ticket price in euro between two cities this weekend.
    Returns the price and whether it is under 100 euro."""
    
    # Mock pricing — replace with Trainline / NS API for real data
    import random
    random.seed(hash(f"{origin}{destination}"))
    price = round(random.uniform(49, 149), 2)

    verdict = "✅ Under budget" if price < 100 else "❌ Over budget"
    return f"Cheapest ticket {origin} → {destination}: €{price}"


# --- Run both tools and reason over the results ---
weather = get_weekend_rain("Paris")
ticket  = get_train_price("Amsterdam", "Paris")

print(weather)
print()
print(ticket)

## Now use this model with ollama

In [ ]:
import ollama

result = ollama.list()
model_names = [model.model for model in result.models]
print(model_names)

In [ ]:
#if you have a decent laptop (min 16GB RAM), you can run a local model like gemma4
!ollama pull gemma4:e4b

In [ ]:
# Set the model to use for the MCP
model = "gemma4:e4b"

In [ ]:
import ollama

# Run tools (same as before)
weather = get_weekend_rain("Paris")
ticket  = get_train_price("Amsterdam", "Paris")

print(weather)
print(ticket)
print()

# Replace anthropic client with ollama
response = ollama.chat(
    model=model,
    messages=[{
        "role": "user",
        "content": f"""
        The user wants to go to Paris this weekend, but only if:
        1. There is no rain
        2. The train from Amsterdam is cheaper than €100

        Weather report: {weather}
        Train ticket: {ticket}

        Should they go? Give a answer based on the weather and ticket information, and explain your reasoning briefly.
        """
    }]
)

print(response.message.content)

### Reflection

Was MCP here better than simply using the search with llama index or smolagents?

Why or why not?